# Text Augmentation Demo

This notebook demonstrates various text augmentation techniques:
1. **Easy Data Augmentation (EDA)**: Synonym replacement, random insertion, swap, deletion
2. **Back-Translation**: Translate to another language and back
3. **LLM Paraphrasing**: Using language models for high-quality paraphrases

**Key Principle**: Text augmentation must preserve the **meaning** and **label**!

In [ ]:
# Install required packages
# !pip install nlpaug nltk transformers torch

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data for synonyms
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("Setup complete!")

## 1. Easy Data Augmentation (EDA)

Four simple operations from Wei & Zou (2019):
- **Synonym Replacement**: Replace words with synonyms
- **Random Insertion**: Insert random synonyms
- **Random Swap**: Swap word positions
- **Random Deletion**: Delete random words

In [ ]:
import nlpaug.augmenter.word as naw

# Sample text for augmentation
original_text = "The movie was absolutely fantastic and I loved every moment of it."
print(f"Original: {original_text}")
print(f"Label: POSITIVE")
print("=" * 70)

In [ ]:
# 1. Synonym Replacement
syn_aug = naw.SynonymAug(aug_src='wordnet')

print("\n🔄 Synonym Replacement:")
for i in range(3):
    augmented = syn_aug.augment(original_text)[0]
    print(f"  {i+1}. {augmented}")

In [ ]:
# 2. Random Insertion
insert_aug = naw.ContextualWordEmbsAug(
    model_path='bert-base-uncased',
    action='insert',
    aug_max=2
)

print("\n➕ Random Insertion (BERT-based):")
for i in range(3):
    augmented = insert_aug.augment(original_text)[0]
    print(f"  {i+1}. {augmented}")

In [ ]:
# 3. Random Swap
swap_aug = naw.RandomWordAug(action='swap', aug_max=2)

print("\n🔀 Random Word Swap:")
for i in range(3):
    augmented = swap_aug.augment(original_text)[0]
    print(f"  {i+1}. {augmented}")

In [ ]:
# 4. Random Deletion
delete_aug = naw.RandomWordAug(action='delete', aug_p=0.1)

print("\n➖ Random Deletion:")
for i in range(3):
    augmented = delete_aug.augment(original_text)[0]
    print(f"  {i+1}. {augmented}")

## 2. Back-Translation

Translate text to another language and back for natural paraphrasing.

In [ ]:
# Back-translation using nlpaug (requires internet)
try:
    back_trans_aug = naw.BackTranslationAug(
        from_model_name='facebook/wmt19-en-de',
        to_model_name='facebook/wmt19-de-en'
    )
    
    print("🔄 Back-Translation (English → German → English):")
    print(f"Original: {original_text}")
    
    for i in range(2):
        augmented = back_trans_aug.augment(original_text)[0]
        print(f"  {i+1}. {augmented}")
        
except Exception as e:
    print(f"Note: Back-translation requires downloading models (~2GB)")
    print(f"Error: {e}")
    print("\nSimulated back-translation examples:")
    print(f"  Original: {original_text}")
    print(f"  1. The film was absolutely wonderful and I enjoyed every moment.")
    print(f"  2. The movie was really fantastic and I loved every second of it.")

## 3. Contextual Word Embeddings (BERT)

Use BERT to suggest contextually appropriate substitutions.

In [ ]:
# BERT-based contextual augmentation
bert_aug = naw.ContextualWordEmbsAug(
    model_path='bert-base-uncased',
    action='substitute',
    aug_max=3
)

print("🤖 BERT Contextual Substitution:")
print(f"Original: {original_text}")
print()

for i in range(3):
    augmented = bert_aug.augment(original_text)[0]
    print(f"  {i+1}. {augmented}")

## 4. ⚠️ Dangerous Augmentations

Some augmentations can **change the label**! Let's see examples.

In [ ]:
# Dangerous: antonym replacement
antonym_aug = naw.AntonymAug()

danger_text = "The service was excellent and the food was delicious."
print("⚠️ DANGEROUS - Antonym Replacement:")
print(f"Original (POSITIVE): {danger_text}")
print()

for i in range(3):
    augmented = antonym_aug.augment(danger_text)[0]
    print(f"  {i+1}. {augmented}")
    print(f"     ❌ Label may have changed!")

In [ ]:
# Dangerous: random word replacement without context
print("\n⚠️ DANGEROUS - Losing Key Words:")
print(f"Original: 'I did NOT like the movie at all'")
print(f"After deletion: 'I did like the movie at all'")
print(f"  ❌ Meaning completely reversed!")

## 5. Named Entity Recognition (NER) Safe Augmentation

When doing NER, we must **protect entity tokens** from modification!

In [ ]:
# NER-safe augmentation: protect named entities
ner_text = "John Smith works at Google in New York City."
protected_words = ['John', 'Smith', 'Google', 'New', 'York', 'City']

# Create augmenter that skips protected words
safe_aug = naw.SynonymAug(
    aug_src='wordnet',
    stopwords=protected_words  # These won't be modified
)

print("✅ NER-Safe Synonym Replacement:")
print(f"Original: {ner_text}")
print(f"Protected: {protected_words}")
print()

for i in range(3):
    augmented = safe_aug.augment(ner_text)[0]
    print(f"  {i+1}. {augmented}")
    # Verify entities preserved
    all_preserved = all(word in augmented for word in ['John', 'Smith', 'Google', 'New York'])
    status = "✅ Entities preserved" if all_preserved else "❌ Entity modified!"
    print(f"     {status}")

## 6. Combining Multiple Augmentations

Chain augmentations for more variety.

In [ ]:
import nlpaug.flow as naf

# Create a pipeline of augmentations
aug_pipeline = naf.Sequential([
    naw.SynonymAug(aug_src='wordnet', aug_p=0.3),
    naw.RandomWordAug(action='swap', aug_p=0.1),
])

sample = "The restaurant had amazing food and great service."

print("🔗 Combined Augmentation Pipeline:")
print(f"Original: {sample}")
print()

for i in range(5):
    augmented = aug_pipeline.augment(sample)[0]
    print(f"  {i+1}. {augmented}")

## 7. Batch Augmentation for Training

Generate multiple augmented samples for training data expansion.

In [ ]:
# Training data expansion
training_samples = [
    ("The product quality is excellent", "POSITIVE"),
    ("Terrible customer service experience", "NEGATIVE"),
    ("Fast shipping and good packaging", "POSITIVE"),
]

aug = naw.SynonymAug(aug_src='wordnet')

print("📊 Expanding Training Data (3 → 12 samples):")
print("=" * 60)

expanded_data = []
for text, label in training_samples:
    expanded_data.append((text, label))  # Keep original
    print(f"\nOriginal [{label}]: {text}")
    
    # Generate 3 augmentations
    for i in range(3):
        aug_text = aug.augment(text)[0]
        expanded_data.append((aug_text, label))
        print(f"  Aug {i+1} [{label}]: {aug_text}")

print(f"\n✅ Expanded from {len(training_samples)} to {len(expanded_data)} samples!")

## Key Takeaways

1. **Synonym replacement** is safe and effective for most tasks
2. **Back-translation** produces natural paraphrases
3. **BERT-based augmentation** is context-aware
4. **Protect named entities** in NER tasks
5. **Never use antonym replacement** - it flips labels!
6. **Always verify** augmented text preserves the intended label